In [ ]:
!python -m pip install --upgrade pip
%pip install pandas matplotlib seaborn scikit-learn openpyxl tensorflow xgboost aif360
%pip install "aif360[Reductions, inFairness]"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pprint import pprint
from collections import Counter
from scipy.stats import chi2_contingency, fisher_exact
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression,                 LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore
from tensorflow.keras.layers import Dense, BatchNormalization # type: ignore

from aif360.datasets import BinaryLabelDataset, StandardDataset
from aif360.algorithms.preprocessing import LFR
from fairlearn.preprocessing import CorrelationRemover
from aif360.algorithms.inprocessing import GerryFairClassifier, PrejudiceRemover, MetaFairClassifier
from aif360.algorithms.postprocessing import EqOddsPostprocessing, RejectOptionClassification

from fairlearn.metrics import MetricFrame
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference, selection_rate, false_positive_rate, false_negative_rate, count
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

random_seed = 15

In [ ]:
PATH = 'C:/Users/aberti/Desktop/ProjectWork_AEQUITAS_AKKODIS-main/'
df = (
    pd.read_excel(PATH + 'data/Dataset_Preprocessed.xlsx')
      .rename(columns=lambda c: c.lstrip().title())
)
df.head()

## Train

### Dataset Preparation

In [ ]:
df = shuffle(df, random_state=random_seed)

X_full = df.drop(columns=['Status_encoded'])
y = df['Status_encoded']
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(X_full, y, test_size=0.2, random_state=random_seed, stratify=y)

X = df.drop(columns=['Status_encoded', 'Sex_encoded'])
y = df['Status_encoded']
s = df['Sex_encoded']
X_train, X_test, y_train, y_test, s_train, s_test = train_test_split(X, y, s, test_size=0.2, random_state=random_seed, stratify=y)

In [ ]:
train_df = X_train.copy()
train_df['target'] = y_train.values
train_df['sex'] = s_train.values

train_ds = StandardDataset(
    train_df,
    label_name='target',
    favorable_classes=[1],
    protected_attribute_names=['sex'],
    privileged_classes=[[1]]
)
train_df.head()

In [ ]:
test_df = X_test.copy()
test_df['target'] = y_test.values
test_df['sex'] = s_test.values

test_ds = StandardDataset(
    test_df,
    label_name='target',
    favorable_classes=[1],
    protected_attribute_names=['sex'],
    privileged_classes=[[1]]
)
test_df.head()

### Pre-Processing

In [ ]:
lfr = LFR(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}],
    k=10, Ax=5, Ay=5, Az=10, verbose=1
)

lfr = lfr.fit(train_ds)

X_train_lfr_df = pd.DataFrame(lfr.transform(train_ds).features, columns=train_ds.feature_names)
X_test_lfr_df = pd.DataFrame(lfr.transform(test_ds).features, columns=train_ds.feature_names)

In [ ]:
clf = LogisticRegression(solver='liblinear')
clf.fit(X_train_lfr_df, y_train)
preds_lfr = clf.predict(X_test_lfr_df)

In [ ]:
cr = CorrelationRemover(sensitive_feature_ids=['sex'])

X_train_cr = cr.fit_transform(train_df)
X_train_cr_df = pd.DataFrame(X_train_cr, columns=train_df.columns.drop("sex"))

X_test_cr = cr.transform(test_df)
X_test_cr_df = pd.DataFrame(X_test_cr, columns=test_df.columns.drop("sex"))


In [ ]:
clf = LogisticRegression(solver='liblinear')
clf.fit(X_train_cr_df, y_train)
preds_lfr = clf.predict(X_test_cr_df)

### In-Processing

In [ ]:
gfc = GerryFairClassifier(
    C=100,
    gamma=0.01,
    fairness_def='FP',
    max_iters=50,
    printflag=False
)
gfc.fit(train_ds)
pred_gfc = gfc.predict(test_ds)

In [ ]:
pr = PrejudiceRemover(sensitive_attr='sex', eta=25.0)
pr.fit(train_ds)
pred_pr = pr.predict(test_ds)

In [ ]:
mfc = MetaFairClassifier(
    sensitive_attr='sex',
    tau=0.5,           # safer default
    type='fdr',        # or 'sr' for statistical rate
    seed=random_seed
)
mfc.fit(train_ds)
pred_mfc = mfc.predict(test_ds)

### Post-Processing

In [ ]:
eop = EqOddsPostprocessing(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}]
)
eop = eop.fit(train_ds, gfc.predict(train_ds))
pred_eop = eop.predict(test_ds)

In [ ]:
roc = RejectOptionClassification(
    unprivileged_groups=[{'sex': 0}],
    privileged_groups=[{'sex': 1}],
    low_class_thresh=0.01,
    high_class_thresh=0.99,
    num_class_thresh=100,
    metric_name='Average odds difference'
)
roc = roc.fit(train_ds, gfc.predict(train_ds))
pred_roc = roc.predict(test_ds)

### Metrics

In [ ]:
def compute_fairness_metrics(y_true, y_pred, sensitive_features, label=None):
    mf = MetricFrame(
        metrics={
            'selection_rate': selection_rate,
            'dp_diff': demographic_parity_difference,
            'eo_diff': equalized_odds_difference,
            'fpr': false_positive_rate,
            'fnr': false_negative_rate,
            'count': count
        },
        y_true=y_true,
        y_pred=y_pred,
        sensitive_features=sensitive_features
    )
    if label:
        print(f"=== {label} ===")
    print(mf.by_group)
    print("Overall:", mf.overall, "\n")
    return mf

In [ ]:
# Pre-Processing
compute_fairness_metrics(y_test, preds_lfr, s_test, label="LFR + LogisticRegression")

# In-processing
compute_fairness_metrics(y_test, pred_gfc.ravel(), s_test, label="GerryFairClassifier")
compute_fairness_metrics(y_test, pred_pr.ravel(), s_test, label="PrejudiceRemover")
compute_fairness_metrics(y_test, pred_mfc.ravel(), s_test, label="MetaFairClassifier")

# Post-processing
compute_fairness_metrics(y_test, pred_eop.ravel(), s_test, label="EqOddsPostprocessing")
compute_fairness_metrics(y_test, pred_roc.ravel(), s_test, label="RejectOptionClassification")

### Models

In [ ]:
def create_model(seed):
    tf.random.set_seed(seed)
    model = Sequential()
    model.add(Dense(128, input_dim=22, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(128, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(64, activation='relu'))
    model.add(BatchNormalization())
    model.add(Dense(1, activation='sigmoid'))

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeClassifier(),
    'Naive Bayes': GaussianNB(),
    'XGBoost': XGBClassifier(),
    'KNN': KNeighborsClassifier(),
    'Neural Network': create_model(random_seed)
}

In [ ]:
metrics = []
predictions = {}

for name, model in models.items():
    if name in ['Linear Regression', 'Decision Tree', 'Naive Bayes', 'XGBoost', 'KNN']:
        model.fit(X_train_full, y_train_full)
    elif name in ['Neural Network']:
        model.fit(X_train_full, y_train_full, epochs=15, batch_size=64, validation_split=0.2)
    else:
        print("Error in Models!"); break

    y_pred = model.predict(X_test_full)

    if name in ['Linear Regression', 'XGBoost', 'Neural Network']:
        y_pred = (y_pred > 0.5).astype(int)

    accuracy = round(accuracy_score(y_test_full, y_pred), 3)
    precision = round(precision_score(y_test_full, y_pred), 3)
    recall = round(recall_score(y_test_full, y_pred), 3)
    f1 = round(f1_score(y_test_full, y_pred), 3)
    roc_auc = round(roc_auc_score(y_test_full, y_pred), 3)

    metrics.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-score': f1,
        'ROC AUC': roc_auc
    })
    predictions[name] = y_pred

metrics = pd.DataFrame(metrics)
predictions_df = pd.DataFrame({
    'Linear Regression' : predictions['Linear Regression'],
    'Decision Tree' : predictions['Decision Tree'],
    'Naive Bayes' : predictions['Naive Bayes'],
    'XGBoost' : predictions['XGBoost'],
    'kNN' : predictions['KNN'],
    'Neural Network' : predictions['Neural Network']
})

## Fairness Metrics

#### **3.1 Demographic Parity**

In [ ]:
sensitive_features = [' Sex_encoded', ' Age Range_encoded', ' Citizenship_encoded', ' Protected category_encoded']
non_sensitive_features = ['Technical Skills', 'Comunication', 'Maturity', 'Dynamism', 'Mobility',
       'English', ' Study area_encoded', ' Study Title_encoded', ' Years Experience_encoded', ' Sector_encoded', ' Job Family Hiring_encoded',
       ' Job Title Hiring_encoded', ' Overall_encoded', ' Years Experience.1_encoded',' Minimum Ral_encoded', ' Ral Maximum_encoded',
       ' Study Level_encoded', 'Current Ral_encoded', 'Expected Ral_encoded']
models_list = [model for model in models]
tolerance = 0.15
significance_level = 0.1

In [ ]:
def calculate_demographic_parity(predictions, sensitive_attribute, name, significance_level, tolerance, activate_check=False):

    df = pd.DataFrame({
        'predictions': predictions,
        'sensitive_attribute': sensitive_attribute
    })
    prop = df.groupby('sensitive_attribute')['predictions'].mean()
    
    if activate_check:
        print(f"===\n{name}\n{prop}")

    if prop.shape[0] == 2:
        return 'T' if (prop.max() - prop.min()) <= tolerance else False
    else:
        contingency_table = pd.crosstab(df['predictions'], df['sensitive_attribute'])
        chi2, p, dof, expected = chi2_contingency(contingency_table)

        if activate_check and (expected < 5).any():
            print(f"Sparse contingency for {name}")
                
        return 'T' if p > significance_level else False
    

table = []
for model in models:
    row = []
    for sensitive_feature in sensitive_features:
        result = calculate_demographic_parity(predictions[model], X_test_full[sensitive_feature], sensitive_feature, significance_level, tolerance, activate_check=True)
        row.append(result)
    table.append(row)
sf_df = pd.DataFrame(table, index = models_list, columns=sensitive_features)

#### **3.2 Equalized odds**

In [ ]:
def calculate_equalized_odds(predictions, true_labels, sensitive_attribute, name, tolerance, activate_check=False):
    df = pd.DataFrame({
        'predictions': predictions,
        'true_labels': true_labels,
        'sensitive_attribute': sensitive_attribute
    })
    tprs, fprs = [], []
    for _, group_df in df.groupby('sens'):
        tn, fp, fn, tp = confusion_matrix(group_df['true_labels'], group_df['predictions'], labels=[0, 1]).ravel()
        tprs.append(tp / (tp + fn) if tp + fn != 0 else 0)
        fprs.append(fp / (fp + tn) if fp + tn != 0 else 0)

    max_tpr_diff = max(tprs) - min(tprs)
    max_fpr_diff = max(fprs) - min(fprs)

    if activate_check:
            print(f"===\n{name}\nMax FPR diff: {max_fpr_diff}\nMax TPR diff: {max_tpr_diff}")

    return 'T' if (max_tpr_diff <= 2 * tolerance and max_fpr_diff <= 2 * tolerance) else False


table = []
for model in models:
    row = []
    for sensitive_feature in sensitive_features:
        result = calculate_equalized_odds(predictions[model], y_test_full, X_test_full[sensitive_feature], sensitive_feature, tolerance, activate_check=False)
        row.append(result)
    table.append(row)
sf_df = pd.DataFrame(table, index = models_list, columns=sensitive_features)